# RecSys Blind A v2 — **local only (Cursor)**

1. One-time: `cp .env.example .env` and put your `HF_TOKEN` in `.env`
2. Select kernel: `music-crs-baselines/.venv/bin/python3`
3. Run cells **1 → 5** top to bottom (no restart, no Colab)

Or in terminal: `./run_local.sh`

In [ ]:
# 1. Paths + check v2
import os, inspect, sys
from pathlib import Path

ROOT = Path("/Users/tahamoula/Desktop/RecSys Competition")
BASE = ROOT / "music-crs-baselines"
CONFIG_TID = "llama1b_bm25_rerank_blindset_A"
BATCH_SIZE = 4  # lower if you run out of memory

os.chdir(BASE)
sys.path.insert(0, str(BASE))

import torch
from mcrs import load_crs_baseline

assert (BASE / "mcrs/reranker_modules/embedding_reranker.py").exists(), "v2 reranker missing"
assert "reranker_type" in str(inspect.signature(load_crs_baseline))

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("ROOT:", ROOT)
print("Device:", device)
print("Config:", CONFIG_TID)
print("v2 OK")

In [ ]:
# 2. Hugging Face login (reads .env in project root)
import os
from pathlib import Path
from huggingface_hub import login, hf_hub_download

env = Path("/Users/tahamoula/Desktop/RecSys Competition/.env")
token = os.environ.get("HF_TOKEN", "")
if not token and env.exists():
    for line in env.read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            token = line.split("=", 1)[1].strip().strip('"').strip("'")

if not token or token == "hf_your_token_here":
    raise ValueError("Create .env with HF_TOKEN=hf_... (copy from .env.example)")

login(token=token)
os.environ["HF_TOKEN"] = token
print("Llama OK:", hf_hub_download("meta-llama/Llama-3.2-1B-Instruct", "config.json", token=token))

In [ ]:
# 3. Patch config for local device
from omegaconf import OmegaConf
from pathlib import Path

p = Path(f"config/{CONFIG_TID}.yaml")
cfg = OmegaConf.load(p)
cfg.device = device
cfg.attn_implementation = "eager"
OmegaConf.save(cfg, p)
print(OmegaConf.to_yaml(cfg))

In [ ]:
# 4. Inference (~30-90 min on Mac MPS, longer on CPU)
from types import SimpleNamespace
from run_inference_blindset import main

main(SimpleNamespace(
    tid=CONFIG_TID,
    eval_dataset="blindset_A",
    batch_size=BATCH_SIZE,
    save_path="./exp/inference",
))

In [ ]:
# 5. Zip for Codabench
import json, zipfile
from pathlib import Path

out = BASE / "exp/inference/blindset_A" / f"{CONFIG_TID}.json"
print("Predictions:", len(json.load(open(out))))

zip_path = ROOT / f"{CONFIG_TID}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(out, arcname="prediction.json")
print("Submit:", zip_path)
print("Contents:", zipfile.ZipFile(zip_path).namelist())

In [ ]:
*(cells below unused — local flow is cells 1–5 only)*

In [ ]:
# 7. Inference (~5-20 min)
from types import SimpleNamespace
from run_inference_blindset import main
main(SimpleNamespace(tid=CONFIG_TID, eval_dataset="blindset_A", batch_size=BATCH_SIZE, save_path="./exp/inference"))

In [ ]:
# 8. Validate + zip for Codabench
import json, zipfile
from pathlib import Path

out = Path(f"exp/inference/blindset_A/{CONFIG_TID}.json")
data = json.load(open(out))
print("Predictions:", len(data))

zip_path = Path(f"{CONFIG_TID}.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(out, arcname="prediction.json")
print("Zip contents:", zipfile.ZipFile(zip_path).namelist())
print("Submit this file to Codabench:", zip_path)